TODO:
- [ ] currently tool assumes first non-filter column it finds is the x-axis. we need the ability to switch between different columns as the x-axis. preferably also a 3D plot so we can see two axes at once. And preferably also sliders so we can move along even more axes

In [ ]:
import marimo as mo
import pandas as pd
import plotly.express as px
import glob
import os

# Benchmark Results Viewer

In [ ]:
plot_titles = [
    "Forward Time (ms)",
    "Backward Time (ms)",
    "Forward Peak Memory (GB)",
    "Backward Peak Memory (GB)",
]

current_dir = os.path.dirname(os.path.abspath(__file__))
csv_files = glob.glob(os.path.join(current_dir, 'bench_results/*.csv'))

if not csv_files:
    mo.md("No benchmark CSV files found in this directory.")

# Create a mapping from a user-friendly name to the file path
csv_options = {os.path.basename(f): f for f in sorted(csv_files)}

In [ ]:
csv_multiselector = mo.ui.multiselect(
    options=csv_options,
    label="Select Benchmark CSV:",
    value=list(csv_options.keys())[:1],
)
csv_multiselector

In [ ]:
if csv_multiselector.value:
    dfs = [pd.read_csv(val) for val in csv_multiselector.value]
else:
    dfs = []

In [ ]:
if not dfs:
    df = pd.DataFrame()
else:
    processed_dfs = []
    for path, single_df in zip(csv_multiselector.value, dfs):
        # Extract module name from filename, e.g., 'MLP_mps.csv' -> 'MLP'
        module_name = os.path.basename(path).split('_')[0]
        single_df['module'] = module_name
        processed_dfs.append(single_df)

    # Concatenate all dataframes. Pandas handles mismatched columns by filling with NaN.
    df = pd.concat(processed_dfs, ignore_index=True)

In [ ]:
if not dfs:
    #x_axis_col_ = None
    x_axis_options = []
    x_axis_dropdown = None
else:
    x_axis_options = [col_ for col_ in df_.columns if df_[col_].dtype not in ['object', 'bool'] and col_ != 'value']

    x_axis_dropdown = mo.ui.dropdown(
        options=x_axis_options,
        label="Select x-axis for measurement:",
        value=x_axis_options[0],
    )
x_axis_dropdown

In [ ]:
if not dfs:
    filters_form_ = mo.md("No CSVs selected. Please select one or more benchmark CSVs from the dropdown above.")
else:
    # Identify columns to create filters for. This will now include 'module'.
    x_axis_cols_not_used_as_axis = [x for x in x_axis_options if x != x_axis_dropdown.value]
    cols_to_filter_ = [
        col_ for col_ in df_.columns 
        if col_ not in ['value', 'measurement', 'module'] and df_[col_].dtype == 'object'
    ] + x_axis_cols_not_used_as_axis

    # Create filters. We explicitly handle NaN values by converting them to a
    # selectable 'N/A' string in the filter options.
    filters_ = {}
    for col_ in cols_to_filter_:
        # Get unique values, filling NaNs with a string 'N/A'
        options_ = sorted(df_[col_].fillna('N/A').unique().tolist(), key=str)
        filters_[col_] = mo.ui.multiselect(
            options=options_,
            label=f"Filter ''{col_}'': ",
            # Default to selecting all available options
            value=options_[:1],
        )

    filters_form_ = mo.md("\n".join([f"{{{col_}}}\n" for col_ in cols_to_filter_])).batch(**filters_).form(show_clear_button=True)
filters_form_

In [ ]:
# Start with a copy of the merged dataframe to apply filters to.
filtered_df__ = df_.copy()

# The `filters_form_.value` holds the current selections from the UI.
# It's a dict like {'column_name': ['value1', 'N/A']}.
for col__, selected_options__ in filters_form_.value.items():

    # Only apply a filter if the user has selected any options for it.
    if selected_options__:

        # Check if the user wants to include rows where this parameter is not applicable.
        include_na__ = 'N/A' in selected_options__

        # Get the list of actual parameter values the user selected.
        standard_options__ = [opt__ for opt__ in selected_options__ if opt__ != 'N/A']

        # Case 1: User selected both 'N/A' and other values.
        if include_na__ and standard_options__:
            # Keep rows where the column's value is in the list OR the value is NaN.
            filtered_df__ = filtered_df__[
                filtered_df__[col__].isin(standard_options__) | filtered_df__[col__].isna()
            ]

        # Case 2: User selected only standard values.
        elif standard_options__:
            filtered_df__ = filtered_df__[filtered_df__[col__].isin(standard_options__)]

        # Case 3: User selected only 'N/A'.
        elif include_na__:
            filtered_df__ = filtered_df__[filtered_df__[col__].isna()]

# The output of this cell is `filtered_df_`.
# It is now correctly filtered and ready for plotting in the next cell.

In [ ]:
# This code assumes `filtered_df_` (the filtered DataFrame) and 
# `x_axis_col_` (the name of the x-axis column) are available from previous cells.
plot_ = None
if filtered_df__.empty or not x_axis_dropdown.value:
    # Use mo.md to display a message if there's nothing to plot.
    mo.md("### No data to plot. Please adjust filters or select different CSVs.")
else:
    # Identify columns to use for creating the plot series. This automatically
    # includes the new 'module' column as well as other parameters.
    series_cols_ = [
        col___
        for col___ in filtered_df__.columns
        if col___ not in ["value", "measurement", x_axis_dropdown.value] and filtered_df__[col___].dtype == "object"
    ]

    plot_df_ = filtered_df__.copy()

    # Create the 'series' column for coloring the plots.
    # It will look like 'MLP-relu-float16', 'GatedMLP-relu-float16', etc.
    # The .astype(str) gracefully handles any lingering 'nan' values for the legend.
    if series_cols_:
        plot_df_["series"] = plot_df_[series_cols_].apply(
            lambda row: "-".join(row.values.astype(str)), axis=1
        )
        color_arg_ = "series"
    else:
        color_arg_ = None

    # --- Plotting Logic (largely unchanged) ---
    color_discrete_map_ = None
    if color_arg_:
        # Sort series names for a consistent legend order
        series_names_ = sorted(plot_df_[color_arg_].unique())
        colors_ = px.colors.qualitative.Plotly
        color_discrete_map_ = {
            series: colors_[i % len(colors_)] for i, series in enumerate(series_names_)
        }

    plots_ = {}
    metrics_ = plot_df_["measurement"].unique()

    for metric_ in plot_titles:
        if metric_ in metrics_:
            metric_df_ = plot_df_[plot_df_["measurement"] == metric_]
            if not metric_df_.empty:
                fig_ = px.line(
                    metric_df_,
                    x=x_axis_dropdown.value, # Use the new x-axis variable
                    y="value",
                    color=color_arg_,
                    title=metric_,
                    markers=True,
                    color_discrete_map=color_discrete_map_,
                )
                fig_.update_layout(
                    margin=dict(l=30, r=30, t=40, b=30), showlegend=False
                )
                plots_[metric_] = fig_

    if not plots_:
        mo.md("No metrics measured for this selection.")
    else:
        # Create a custom legend
        legend_items_ = []
        if color_discrete_map_:
            for series_, color_ in color_discrete_map_.items():
                legend_items_.append(
                    mo.md(
                        f"""
                        <div style="display: flex; align-items: center; margin-right: 15px; margin-bottom: 5px;">
                            <div style="width: 12px; height: 12px; background-color: {color_}; margin-right: 5px; border-radius: 2px;"></div>
                            <span style="font-size: 0.9em;">{series_}</span>
                        </div>
                        """
                    )
                )
        custom_legend_ = mo.hstack(legend_items_, justify="center", wrap=True)

        # Display plots in a 2x2 grid, handling missing plots gracefully
        row1_ = mo.hstack(
            [plots_.get(plot_titles[0]), plots_.get(plot_titles[1])], justify="center"
        )
        row2_ = mo.hstack(
            [plots_.get(plot_titles[2]), plots_.get(plot_titles[3])], justify="center"
        )
        plot_ = mo.vstack([custom_legend_, row1_, row2_], align="center")
plot_

relu-bfloat16-MLP-mps-channel 
 <marimo-plotly data-figure='{"data": [{"hovertemplate": "series=relu-bfloat16-MLP-mps-channel<br>dim=%{x}<br>value=%{y}<extra></extra>", "legendgroup": "relu-bfloat16-MLP-mps-channel", "line": {"color": "#636EFA", "dash": "solid"}, "marker": {"symbol": "circle"}, "mode": "lines+markers", "name": "relu-bfloat16-MLP-mps-channel", "orientation": "v", "showlegend": true, "x": {"dtype": "i2", "bdata": "gACAAIAAAAIAAgACAAQABAAEAAgACAAI"}, "xaxis": "x", "y": {"dtype": "f8", "bdata": "AADgkhiE0z///39N0ZHVPwAAkJXUidw/AAC8WyAh8D8AAHQGQWz+PwAA7o0VVg1AAAALjYgwGEAAgKGyDLklQAAAqOERfTRAAABqb/DsRkAAABWDLBFGQABAXphxvFNA"}, "yaxis": "y", "type": "scatter"}], "layout": {"template": {"data": {"histogram2dcontour": [{"type": "histogram2dcontour", "colorbar": {"outlinewidth": 0, "ticks": ""}, "colorscale": [[0.0, "#0d0887"], [0.1111111111111111, "#46039f"], [0.2222222222222222, "#7201a8"], [0.3333333333333333, "#9c179e"], [0.4444444444444444, "#bd3786"], [0.5555555555555556, "#d8576b"], [0.6666666666666666, "#ed7953"], [0.7777777777777778, "#fb9f3a"], [0.8888888888888888, "#fdca26"], [1.0, "#f0f921"]]}], "choropleth": [{"type": "choropleth", "colorbar": {"outlinewidth": 0, "ticks": ""}}], "histogram2d": [{"type": "histogram2d", "colorbar": {"outlinewidth": 0, "ticks": ""}, "colorscale": [[0.0, "#0d0887"], [0.1111111111111111, "#46039f"], [0.2222222222222222, "#7201a8"], [0.3333333333333333, "#9c179e"], [0.4444444444444444, "#bd3786"], [0.5555555555555556, "#d8576b"], [0.6666666666666666, "#ed7953"], [0.7777777777777778, "#fb9f3a"], [0.8888888888888888, "#fdca26"], [1.0, "#f0f921"]]}], "heatmap": [{"type": "heatmap", "colorbar": {"outlinewidth": 0, "ticks": ""}, "colorscale": [[0.0, "#0d0887"], [0.1111111111111111, "#46039f"], [0.2222222222222222, "#7201a8"], [0.3333333333333333, "#9c179e"], [0.4444444444444444, "#bd3786"], [0.5555555555555556, "#d8576b"], [0.6666666666666666, "#ed7953"], [0.7777777777777778, "#fb9f3a"], [0.8888888888888888, "#fdca26"], [1.0, "#f0f921"]]}], "contourcarpet": [{"type": "contourcarpet", "colorbar": {"outlinewidth": 0, "ticks": ""}}], "contour": [{"type": "contour", "colorbar": {"outlinewidth": 0, "ticks": ""}, "colorscale": [[0.0, "#0d0887"], [0.1111111111111111, "#46039f"], [0.2222222222222222, "#7201a8"], [0.3333333333333333, "#9c179e"], [0.4444444444444444, "#bd3786"], [0.5555555555555556, "#d8576b"], [0.6666666666666666, "#ed7953"], [0.7777777777777778, "#fb9f3a"], [0.8888888888888888, "#fdca26"], [1.0, "#f0f921"]]}], "surface": [{"type": "surface", "colorbar": {"outlinewidth": 0, "ticks": ""}, "colorscale": [[0.0, "#0d0887"], [0.1111111111111111, "#46039f"], [0.2222222222222222, "#7201a8"], [0.3333333333333333, "#9c179e"], [0.4444444444444444, "#bd3786"], [0.5555555555555556, "#d8576b"], [0.6666666666666666, "#ed7953"], [0.7777777777777778, "#fb9f3a"], [0.8888888888888888, "#fdca26"], [1.0, "#f0f921"]]}], "mesh3d": [{"type": "mesh3d", "colorbar": {"outlinewidth": 0, "ticks": ""}}], "scatter": [{"fillpattern": {"fillmode": "overlay", "size": 10, "solidity": 0.2}, "type": "scatter"}], "parcoords": [{"type": "parcoords", "line": {"colorbar": {"outlinewidth": 0, "ticks": ""}}}], "scatterpolargl": [{"type": "scatterpolargl", "marker": {"colorbar": {"outlinewidth": 0, "ticks": ""}}}], "bar": [{"error_x": {"color": "#2a3f5f"}, "error_y": {"color": "#2a3f5f"}, "marker": {"line": {"color": "#E5ECF6", "width": 0.5}, "pattern": {"fillmode": "overlay", "size": 10, "solidity": 0.2}}, "type": "bar"}], "scattergeo": [{"type": "scattergeo", "marker": {"colorbar": {"outlinewidth": 0, "ticks": ""}}}], "scatterpolar": [{"type": "scatterpolar", "marker": {"colorbar": {"outlinewidth": 0, "ticks": ""}}}], "histogram": [{"marker": {"pattern": {"fillmode": "overlay", "size": 10, "solidity": 0.2}}, "type": "histogram"}], "scattergl": [{"type": "scattergl", "marker": {"colorbar": {"outlinewidth": 0, "ticks": ""}}}], "scatter3d": [{"type": "scatter3d", "line": {"colorbar": {"outlinewidth": 0, "ticks": ""}}, "